In [1]:
import pandas as pd
import numpy as np

# =========================================================
# READ SOURCE DATA
# =========================================================
pvt_df = pd.read_excel("Pvt draft.xlsx")
ptc_df = pd.read_excel("PTC draft 2025-12-24.xlsx")

# =========================================================
# NORMALIZE PN (ALPHANUMERIC SAFE)
# =========================================================
pvt_df["PN AL78"] = pvt_df["PN AL78"].astype(str).str.strip()
ptc_df["PN AL78"] = ptc_df["PN AL78"].astype(str).str.strip()

# =========================================================
# AGGREGATION: OH BY PN AL78
# =========================================================
oh_df = (
    pvt_df
    .groupby("PN AL78", as_index=False)
    .agg({"OH": "mean"})
    .rename(columns={"OH": "Avrg.OH CM"})
)

# =========================================================
# AGGREGATION: OTHERS BY PN NO SUFFIX
# =========================================================
main_df = (
    pvt_df
    .groupby("PN.Final", as_index=False)
    .agg({
        "PN used": "first",
        "Description": "first",
        "In Rcv Total": "mean",
        "Qty Resvered": "sum",
        "In Tr Total": "mean",
        "7 Months Prior": "sum",
        "OO CM": "sum",
        "OO NM": "sum",
        "OO N2M": "sum",
        "Curr. Month": "max",
        "Req CM": "max",
        "Req NM": "max",
        "PN AL78": "first"
    })
)

# =========================================================
# MERGE OH
# =========================================================
result_df = main_df.merge(oh_df, on="PN AL78", how="left")

# =========================================================
# RENAME COLUMNS
# =========================================================
result_df = result_df.rename(columns={
    "PN.Final": "PN No suffix",
    "PN used": "Long PN",
    "In Rcv Total": "Avrg. InRcv",
    "Qty Resvered": "Sum Qty Resvrd",
    "In Tr Total": "Avrg. InTr",
    "7 Months Prior": "Sum 7 Months Prior",
    "OO CM": "Sum OO CM",
    "OO NM": "Sum OO NM",
    "OO N2M": "Sum OO N2M",
    "Curr. Month": "Max.Curr. Month",
    "Req CM": "Max.Req. CM",
    "Req NM": "Max.Req. NM",
})

# =========================================================
# ADD 6 EMPTY COLUMNS
# =========================================================
empty_cols = [f"Empty_{i}" for i in range(1, 7)]
insert_pos = result_df.columns.get_loc("Max.Req. NM") + 1

for i, col in enumerate(empty_cols):
    result_df.insert(insert_pos + i, col, np.nan)

# =========================================================
# FIX ESD MON TYPE
# =========================================================
ptc_df["ESD Mon"] = pd.to_numeric(ptc_df["ESD Mon"], errors="coerce").astype("Int64")

# =========================================================
# PTC AGGREGATION (CORRECT & SINGLE LOGIC)
# =========================================================

# Detect ESD months dynamically
esd_months = (
    ptc_df["ESD Mon"]
    .dropna()
    .astype(int)
    .sort_values()
    .unique()
    .tolist()
)

# Aggregate Qty Reserved
ptc_sum = (
    ptc_df
    .groupby(["PN AL78", "ESD Mon"], as_index=False)
    .agg(Qty_Resvrd=("Qty Resvered", "sum"))
)

# Pivot
ptc_pivot = (
    ptc_sum
    .pivot(index="PN AL78", columns="ESD Mon", values="Qty_Resvrd")
    .reset_index()
)

# Ensure all month columns exist
for m in esd_months:
    if m not in ptc_pivot.columns:
        ptc_pivot[m] = 0

ptc_pivot = ptc_pivot[["PN AL78"] + esd_months]

# =========================================================
# GRAND TOTAL + MISSING
# =========================================================
ptc_pivot["Grand Total"] = ptc_pivot[esd_months].sum(axis=1)

ptc_pivot["Missing"] = np.where(
    ptc_pivot["Grand Total"] == 0,
    0,
    ""
)

# Display blanks LAST
ptc_pivot[esd_months] = ptc_pivot[esd_months].replace(0, "")

# =========================================================
# MERGE BACK INTO RESULT
# =========================================================
result_df = result_df.merge(ptc_pivot, on="PN AL78", how="left")

# =========================================================
# FINAL COLUMN ORDER
# =========================================================
base_cols = [
    "PN AL78",
    "Long PN",
    "PN No suffix",
    "Description",
    "Avrg.OH CM",
    "Avrg. InRcv",
    "Sum Qty Resvrd",
    "Avrg. InTr",
    "Sum 7 Months Prior",
    "Sum OO CM",
    "Sum OO NM",
    "Sum OO N2M",
    "Max.Curr. Month",
    "Max.Req. CM",
    "Max.Req. NM",
]

esd_cols = esd_months + ["Missing", "Grand Total"]

result_df = result_df[base_cols + empty_cols + esd_cols]


In [2]:
# =========================================================
# ===== ADD BLANK COLUMNS + chk. Avail. (PANDAS) ==========
# =========================================================

insert_pos = result_df.columns.get_loc("Grand Total") + 1

# ---- Add 2 blank columns ----
result_df.insert(insert_pos, " ", np.nan)
result_df.insert(insert_pos + 1, "  ", np.nan)

# ---- Calculate chk. Avail. directly ----
supply = (
    result_df["Avrg.OH CM"].fillna(0)
    + result_df["Avrg. InRcv"].fillna(0)
    + result_df["Avrg. InTr"].fillna(0)
    + result_df["Sum 7 Months Prior"].fillna(0)
    + result_df["Sum OO CM"].fillna(0)
)


result_df.insert(
    insert_pos + 2,
    "chk. Avail.",
    np.where(
        result_df["Avrg.OH CM"] < result_df["Max.Curr. Month"],
        np.where(
            result_df["Max.Curr. Month"] <= supply,
            "OK",
            "NG"
        ),
        "OK"
    )
)


In [3]:
# =========================================================
# ===== ADD Balnc. COLUMN (PANDAS) ========================
# =========================================================

supply_cols = [
    "Avrg.OH CM",
    "Avrg. InRcv",
    "Avrg. InTr",
    "Sum 7 Months Prior",
    "Sum OO CM",
]

supply = result_df[supply_cols].fillna(0).sum(axis=1)

result_df["Balnc."] = supply - result_df["Max.Curr. Month"].fillna(0)


In [4]:
# =========================================================
# ===== ADD chk Resvrd COLUMN (PANDAS) ====================
# =========================================================

result_df.insert(
    result_df.columns.get_loc("Balnc.") + 1,
    "chk Resvrd",
    np.where(
        result_df["Grand Total"].fillna(0)
        == result_df["Sum Qty Resvrd"].fillna(0),
        "OK",
        "NG"
    )
)


In [5]:
ptc_df.groupby(["PN AL78", "Seqnc"]).agg(
    esd=("ESD Mon", "first"),
    qty=("Qty Resvered", "sum")
).loc["4348847"]


,esd,qty
Seqnc,,
1,12,1
2,12,2


In [6]:
print(result_df)

          PN AL78      Long PN  PN No suffix           Description  \
0          107460    010746000        107460                  CLIP   
1          107695    010769500        107695              PIN,ROLL   
2          108722    010872200        108722                  CLIP   
3          109319    010931900        109319             LOCKPLATE   
4          109660    010966000        109660     NIPPLE,PLAIN PIPE   
..            ...          ...           ...                   ...   
981       S   658    S00065800       S   658          WASHER,PLAIN   
982       S   962    S00096200       S   962             PLUG,PIPE   
983  S   962    E  S00096200 E  S   962    E             DRAINCOCK   
984       S  1031    S00103100       S  1031  ELBOW,FEMALE ADAPTER   
985       S  2268    S00226800       S  2268        FITTING,GREASE   

     Avrg.OH CM  Avrg. InRcv  Sum Qty Resvrd  Avrg. InTr  Sum 7 Months Prior  \
0          23.0          0.0              25         8.0                   0   

In [7]:
# # =========================================================
# # EXPORT
# # =========================================================
# result_df.to_excel("Pvt Final draft.xlsx", index=False)